# Build enriched dataset
This notebook does two things:
1. Computes **genre popularity** per market per year from the Charts + Artists data
2. Builds the **enriched model dataset** by joining genre networks + distances + genre popularity

## Config & imports

In [1]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub/Bachelor


In [4]:
import sys
sys.path.append('embeddings')

import os
import ast
import glob
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import torch
from lorentz import Lorentz

MGD_ROOT = 'datasets'  

CHARTS_DIR = os.path.join(MGD_ROOT, 'Charts')
ARTISTS_CSV = os.path.join(MGD_ROOT, 'Artists', 'spotify_artists_info_complete.csv')
GENRE_NET_DIR = os.path.join(MGD_ROOT, 'Original')


CKPT = 'ckpt/final_enao_graph.ckpt'
VOCAB = 'enao_vocab.pkl' 
DIM = 2

MARKETS = ['au', 'br', 'ca', 'de', 'fr', 'gb', 'global', 'jp', 'us']
YEARS = [2017, 2018, 2019]

# output
OUTPUT_POPULARITY = 'prediction_model/genre_popularity.csv'
OUTPUT_DATASET = 'prediction_model/model_dataset.csv'

## 1. Load embeddings

In [5]:
genres_list = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres_list)
genre_to_idx = {g: i for i, g in enumerate(genres_list)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

lorentz_table = net.table.weight.data.cpu().numpy()

def lorentz_distance(u, v):
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    return lorentz_table[genre_to_idx[genre_name] + 1]

print(f'Loaded {n_items} genre embeddings')

Loaded 6280 genre embeddings


## 2. Build genre popularity per market per year

For each (market, year) we sum the weekly streams of every chart song, attributed to each genre of that song's artist.

```
genre_popularity(genre, market, year) =
    sum of streams across all weeks
    for all songs whose artist has that genre
```

In [6]:
artists_df = pd.read_csv(ARTISTS_CSV, sep='\t')

# parse genres from string representation of list
def parse_genres(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

artists_df['genres_parsed'] = artists_df['genres'].apply(parse_genres)

# build name -> genres lookup
artist_genres = dict(zip(artists_df['name'].str.strip(),
                         artists_df['genres_parsed']))

print(f'Artists loaded: {len(artist_genres)}')
print('Sample:', list(artist_genres.items())[:3])

Artists loaded: 3584
Sample: [('AURORA', ['norwegian pop']), ('MHD', ['francoton', 'french hip hop', 'pop urbaine']), ('Chuck Berry', ['blues rock', 'classic rock', 'folk rock', 'rock', 'rock-and-roll', 'rockabilly', 'roots rock', 'soul'])]


In [8]:
from collections import defaultdict

popularity_rows = []
skipped_artists = set()

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        weekly_files = glob.glob(os.path.join(charts_path, '*.csv'))
        if not weekly_files:
            continue

        genre_streams = defaultdict(float)

        for fpath in weekly_files:
            try:
                chart = pd.read_csv(fpath, sep='\t')
            except Exception:
                continue

            for _, row in chart.iterrows():
                artist_name = str(row['artist']).strip()
                streams = row['streams']
                genres = artist_genres.get(artist_name, [])

                if not genres:
                    skipped_artists.add(artist_name)
                    continue

                for genre in genres:
                    genre_streams[genre] += streams

        for genre, total_streams in genre_streams.items():
            popularity_rows.append({
                'market':       market,
                'year':         year,
                'genre':        genre,
                'total_streams': total_streams
            })


popularity_df = pd.DataFrame(popularity_rows)
popularity_df.to_csv(OUTPUT_POPULARITY, index=False)

print(f'\nGenre popularity table: {len(popularity_df)} rows')
print(f'Artists with no genres (skipped): {len(skipped_artists)}')
print()
print(popularity_df[popularity_df['market'] == 'us'][popularity_df['year'] == 2019]
      .sort_values('total_streams', ascending=False).head(10).to_string(index=False))


Genre popularity table: 6258 rows
Artists with no genres (skipped): 131

market  year            genre  total_streams
    us  2019              pop   1.201314e+10
    us  2019              rap   1.070258e+10
    us  2019          pop rap   7.501231e+09
    us  2019      melodic rap   7.103647e+09
    us  2019             trap   5.737092e+09
    us  2019    post-teen pop   5.307230e+09
    us  2019        dance pop   4.593115e+09
    us  2019          hip hop   3.516477e+09
    us  2019       electropop   2.736183e+09
    us  2019 southern hip hop   2.354967e+09


/var/folders/4x/l0sbrhlx0kx4f0hwzx7x2yqr0000gn/T/ipykernel_27393/201896912.py:52: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(popularity_df[popularity_df['market'] == 'us'][popularity_df['year'] == 2019]


## 2b. Compute ranking-based popularity

In addition to stream-based popularity, we compute ranking-based metrics:
- `avg_ranking`: Average chart position (lower = better, e.g., avg rank 10)
- `ranking_score`: Weighted score where rank 1 = 200 points, rank 200 = 1 point (higher = better)
- `n_chart_appearances`: How many weeks the genre appeared on charts

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE RANKING-BASED POPULARITY (in addition to streams)
# ══════════════════════════════════════════════════════════════════════════════

ranking_rows = []

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        # Collect rankings for each genre
        genre_rankings = defaultdict(list)
        
        week_files = sorted(glob.glob(os.path.join(charts_path, '*.csv')))
        skipped_files = 0
        for week_file in week_files:
            try:
                week_df = pd.read_csv(week_file)
                
                # Check for required columns (different files may have different column names)
                # Common variations: 'Artist' or 'artist', 'Position' or 'position' or 'Rank'
                artist_col = None
                position_col = None
                
                for col in week_df.columns:
                    if col.lower() in ['artist', 'artist name', 'track artist']:
                        artist_col = col
                    if col.lower() in ['position', 'rank', 'pos']:
                        position_col = col
                
                if artist_col is None or position_col is None:
                    skipped_files += 1
                    continue
                    
            except Exception as e:
                # Skip files with parsing errors
                skipped_files += 1
                continue
            
            for _, row in week_df.iterrows():
                artist_name = row[artist_col].strip()
                rank = row[position_col]  # 1-200
                
                if artist_name not in artist_genres:
                    continue
                    
                for genre in artist_genres[artist_name]:
                    genre_rankings[genre].append(rank)
        
        if skipped_files > 0:
            print(f'  {market} {year}: skipped {skipped_files} problematic files')
        
        # Compute metrics for each genre
        for genre, ranks in genre_rankings.items():
            avg_rank = np.mean(ranks)
            
            # Weighted score: invert ranking (200-rank+1) and sum
            # Rank 1 = 200 pts, Rank 200 = 1 pt
            weighted_score = sum(200 - r + 1 for r in ranks)
            
            ranking_rows.append({
                'market': market,
                'year': year,
                'genre': genre,
                'avg_ranking': avg_rank,           # Lower = better (avg position)
                'ranking_score': weighted_score,    # Higher = better (total points)
                'n_chart_appearances': len(ranks)   # How many weeks on charts
            })
    
    print(f'{market} ranking data done')

# Create ranking DataFrame
ranking_df = pd.DataFrame(ranking_rows)

print(f'\nGenre ranking table: {len(ranking_df)} rows')
print('\nSample (US 2019, top by ranking score):')
print(ranking_df[(ranking_df['market'] == 'us') & (ranking_df['year'] == 2019)]
      .sort_values('ranking_score', ascending=False).head(10).to_string(index=False))

# Save ranking data separately
OUTPUT_RANKING = 'prediction_model/genre_ranking.csv'
ranking_df.to_csv(OUTPUT_RANKING, index=False)
print(f'\nSaved ranking data to: {OUTPUT_RANKING}')

  au 2017: skipped 52 problematic files
  au 2018: skipped 52 problematic files
  au 2019: skipped 52 problematic files
au ranking data done
  br 2017: skipped 52 problematic files
  br 2018: skipped 52 problematic files
  br 2019: skipped 52 problematic files
br ranking data done
  ca 2017: skipped 52 problematic files
  ca 2018: skipped 52 problematic files
  ca 2019: skipped 52 problematic files
ca ranking data done
  de 2017: skipped 52 problematic files
  de 2018: skipped 52 problematic files
  de 2019: skipped 52 problematic files
de ranking data done
  fr 2017: skipped 52 problematic files
  fr 2018: skipped 52 problematic files
  fr 2019: skipped 52 problematic files
fr ranking data done
  gb 2017: skipped 52 problematic files
  gb 2018: skipped 52 problematic files
  gb 2019: skipped 52 problematic files
gb ranking data done
  global 2017: skipped 50 problematic files
  global 2018: skipped 52 problematic files
  global 2019: skipped 52 problematic files
global ranking data do

KeyError: 'market'

## 3. Build the enriched model dataset

For each (market, year) genre network file we:
- Remove self-loops
- Compute hyperbolic distance for each pair
- Attach `popularity_source` and `popularity_target` from the popularity table
- Add `market` and `year` columns

Then stack all markets and years into one CSV.

In [ ]:
pop_lookup = {}
for _, row in popularity_df.iterrows():
    pop_lookup[(row['market'], row['year'], row['genre'])] = row['total_streams']

# NEW: Ranking lookups
rank_lookup = {}
rank_score_lookup = {}
for _, row in ranking_df.iterrows():
    rank_lookup[(row['market'], row['year'], row['genre'])] = row['avg_ranking']
    rank_score_lookup[(row['market'], row['year'], row['genre'])] = row['ranking_score']

all_rows = []
skipped_no_embedding = 0
skipped_no_popularity = 0

for market in MARKETS:
    for year in YEARS:
        net_path = os.path.join(GENRE_NET_DIR, market,
                                f'{market}-genre_network-{year}.csv')
        if not os.path.exists(net_path):
            print(f'  Missing genre network: {net_path}')
            continue

        gn = pd.read_csv(net_path, sep='\t')
        gn = gn[gn['source'] != gn['target']]  # remove self-loops

        for _, row in gn.iterrows():
            src, tgt = row['source'], row['target']

            # hyperbolic distance
            u = get_embedding(src)
            v = get_embedding(tgt)
            if u is None or v is None:
                skipped_no_embedding += 1
                continue

            dist = lorentz_distance(u, v)

            # genre popularity (streams-based)
            pop_src = pop_lookup.get((market, year, src))
            pop_tgt = pop_lookup.get((market, year, tgt))

            if pop_src is None or pop_tgt is None:
                skipped_no_popularity += 1
                continue

            # NEW: Genre ranking metrics
            rank_src = rank_lookup.get((market, year, src))
            rank_tgt = rank_lookup.get((market, year, tgt))
            rank_score_src = rank_score_lookup.get((market, year, src))
            rank_score_tgt = rank_score_lookup.get((market, year, tgt))

            all_rows.append({
                'market':       market,
                'year':         year,
                'source':       src,
                'target':       tgt,
                'weight':       row['weight'],
                'avg_streams':  row['avg_streams'],
                'distance':     dist,
                
                # Streams-based popularity
                'popularity_source':   pop_src,
                'popularity_target':   pop_tgt,
                
                # NEW: Ranking-based popularity
                'ranking_source':      rank_src,
                'ranking_target':      rank_tgt,
                'ranking_score_source': rank_score_src,
                'ranking_score_target': rank_score_tgt,
                
                # Log transformations (streams)
                'log_streams':  np.log10(row['avg_streams'] + 1),
                'log_popularity_source':  np.log10(pop_src + 1),
                'log_popularity_target':  np.log10(pop_tgt + 1),
                'log_weight':   np.log10(row['weight'] + 1),
                
                # NEW: Log transformations (ranking)
                'log_ranking_score_source': np.log10(rank_score_src + 1) if rank_score_src else None,
                'log_ranking_score_target': np.log10(rank_score_tgt + 1) if rank_score_tgt else None,
            })

model_df = pd.DataFrame(all_rows)

# Add normalized distances (min-max per market for fair cross-market comparison)
print('\nNormalizing distances...')

# Normalize per market
model_df['distance_norm'] = 0.0
for market in model_df['market'].unique():
    mask = model_df['market'] == market
    scaler = MinMaxScaler()
    model_df.loc[mask, 'distance_norm'] = scaler.fit_transform(
        model_df.loc[mask, ['distance']]
    ).flatten()

# Also add global normalization for overall analysis
scaler_global = MinMaxScaler()
model_df['distance_norm_global'] = scaler_global.fit_transform(
    model_df[['distance']]
).flatten()

print(f'Added normalized distances (per-market and global)')
print(f'  Per-market range: {model_df["distance_norm"].min():.4f} - {model_df["distance_norm"].max():.4f}')
print(f'  Global range: {model_df["distance_norm_global"].min():.4f} - {model_df["distance_norm_global"].max():.4f}')
print()

model_df.to_csv(OUTPUT_DATASET, index=False)

print(f'Model dataset: {len(model_df)} rows')
print(f'Skipped (no embedding): {skipped_no_embedding}')
print(f'Skipped (no popularity): {skipped_no_popularity}')
print()
print('Rows per market:')
print(model_df['market'].value_counts().to_string())
print()
print('Rows per year:')
print(model_df['year'].value_counts().sort_index().to_string())

No popularity (13,067 rows skipped). These genres DO exist in the Artists file, but the artists who play them never appeared in the top 200 charts for that specific market/year. For example album rock artists like John Lennon and Journey appear in the US charts, but not in Germany or Brazil. So album rock has zero streams in those markets — it genuinely has no popularity there.

## 4. Quick sanity check

In [ ]:
print('=== Dataset summary ===')
print(model_df[['avg_streams', 'distance', 'popularity_source', 'popularity_target', 
                'ranking_score_source', 'ranking_score_target', 'weight']]
      .describe().round(2))
print()
print('=== Sample rows ===')
print(model_df[['market', 'year', 'source', 'target',
                'distance', 'log_popularity_source', 'log_popularity_target',
                'log_ranking_score_source', 'log_ranking_score_target',
                'log_weight', 'log_streams']]
      .head(10).to_string(index=False))
print()
print('=== Missing values ===')
print(model_df.isnull().sum())
print()
print('=== Ranking vs Streams correlation ===')
# Check correlation between ranking-based and streams-based popularity
print(f"Correlation (ranking_score_source, popularity_source): "
      f"{model_df[['ranking_score_source', 'popularity_source']].corr().iloc[0,1]:.4f}")
print(f"Correlation (ranking_score_target, popularity_target): "
      f"{model_df[['ranking_score_target', 'popularity_target']].corr().iloc[0,1]:.4f}")